In [33]:
import pandas as pd
from geopy.distance import geodesic
from pyprojroot import here

In [34]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [35]:
cent = pd.read_csv(INTERIM_DATA_DIR/"municipality_centroids.csv")
ports_airports = pd.read_csv(RAW_DATA_DIR/"logistics/ports_csv/puertos_aero.csv")

In [36]:
ports = ports_airports[ports_airports["Tipo"] == "Puerto"].copy()

In [37]:
ports.head()

,Tipo,Nombre,Ciudad,Estado,Latitud,Longitud
0,Puerto,Puerto de Manzanillo,Manzanillo,Colima,19.0522,-104.3161
1,Puerto,Puerto Lázaro Cárdenas,Lázaro Cárdenas,Michoacán,17.9472,-102.1764
2,Puerto,Puerto de Veracruz,Veracruz,Veracruz,19.2022,-96.1331
3,Puerto,Puerto de Altamira,Altamira,Tamaulipas,22.4283,-97.9358
4,Puerto,Puerto de Ensenada,Ensenada,Baja California,31.8594,-116.6231


In [38]:
for _, port in ports.iterrows():
    name = port["Nombre"]
    p_coords = (port["Latitud"], port["Longitud"])
    cent[name] = cent.apply(
        lambda row: geodesic((row["lat"], row["long"]), p_coords).km, 
        axis=1
    )

In [39]:
port_cols = ports["Nombre"].tolist()
cent["dist_min_ports"] = cent[port_cols].min(axis=1)
result = cent[["CVEGEO"] + ["dist_min_ports"]]

In [41]:
result.to_csv(INTERIM_DATA_DIR/"ports_dist_by_municipality.csv", index=False)

print(f"Done. {len(result)} municipalities written.")
print(result.sample(10))

Done. 2478 municipalities written.
      CVEGEO  dist_min_ports
1766   21089      205.171841
2353   31022      163.837882
2122   30197       98.131223
705    15024      309.268797
1793   21082      196.150511
601    14060      267.895241
1928   26069      196.853971
1317   20109      159.695369
2176   30206       10.218574
365    11041      263.521420
